**Connect Scripts**

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from Scripts import FileHandler as fh

**Load Data Set**

In [ ]:
# load data set
import kagglehub
from pathlib import Path

downloadPath = kagglehub.dataset_download('lespin/house-prices-dataset')
dataPath = Path(downloadPath)

print(f'Content of {dataPath}:')
for item in dataPath.iterdir():
    print(f"    -{item.name} ({'Folder' if item.is_dir() else 'File'})")

**Inspect Data Set**

In [ ]:
# Inspect data set
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

dfHousing = pd.read_csv(f'{downloadPath}/train.csv')

# print first 5 lines
display(dfHousing.head())

# check data set shape
print(f"\n Housing Dataset shape : {dfHousing.shape}")

# check available data types
print("\n Data Type Count")
print(dfHousing.dtypes.value_counts())

# check missing values in data set
print("\n Missing values in Data set")
dfHousing.info()

# statistical summary
print("\n statistical summary")
display(dfHousing['SalePrice'].describe())

# check distribution
sns.set_theme(style='whitegrid')
plt.figure(figsize=(8,5))
sns.histplot(dfHousing['SalePrice'], kde=True, color='green', bins=30)
plt.title('Distribution of House sale prices')
plt.xlabel('Sale Price/($)')
plt.ylabel('Count')
plt.show()


**Train/Validate/Test Split**
- avoid cross contamination (don't need to get influence by Training data)
- 70% : Train ; 15% : Validation ; 15% : Test

In [ ]:
from sklearn.model_selection import train_test_split

dfTrain, dfTemp = train_test_split(dfHousing, test_size=0.30, random_state=42)
dfVal, dfTest = train_test_split(dfTemp, test_size=0.5, random_state=42)

print(f"Dataset Size: {len(dfHousing)} | Train Size: {len(dfTrain)} | Validate Size: {len(dfVal)} | Test Size: {len(dfTest)}")
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "01_datasplit")

**Handle Missing Values**
- Drop the columns
- Imputation/ Replace
    - mean
    - meadian
    - frequent
    - null value

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing", "01_datasplit")

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
# drop columns
dropList = []

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfVal.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
# replace with median value
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
medianReplaceList = ['LotFrontage']

for column in medianReplaceList:
    if column in dfTrain.columns:
        imputer.fit(dfTrain[[column]])
        print(f"Imputed(median) value: {imputer.statistics_}")

        dfTrain[[column]] = imputer.transform(dfTrain[[column]])

        if column in dfVal:
            dfVal[[column]] = imputer.transform(dfVal[[column]])

        if column in dfTest:
            dfTest[[column]] = imputer.transform(dfTest[[column]])

print("missing in Train: ", dfTrain['LotFrontage'].isnull().sum())

In [ ]:
# replace with frequent value

frequentReplaceList = ['Electrical']

for column in frequentReplaceList:
    if column in dfTrain.columns:
        modeValue = dfTrain[column].mode()[0]
        print(f"Imputed(frequent) value: {modeValue}")

        dfTrain[column] = dfTrain[column].fillna(modeValue)

        if column in dfVal:
            dfVal[column] = dfVal[column].fillna(modeValue)

        if column in dfTest:
            dfTest[column] = dfTest[column].fillna(modeValue)


print("missing in Train: ", dfTrain['Electrical'].isnull().sum())

In [ ]:
catCols = dfTrain.select_dtypes(include=['str']).columns
numCols = dfTrain.select_dtypes(exclude=['str']).columns

catNullList = dfTrain[catCols].columns[dfTrain[catCols].isnull().any()].tolist()
numNullList = dfTrain[numCols].columns[dfTrain[numCols].isnull().any()].tolist()

print(catNullList)
print(numNullList)


In [ ]:
# replace with none/0 value

# fill numerical columns with 0
dfTrain[numNullList] = dfTrain[numNullList].fillna(0)
dfVal[numNullList] = dfVal[numNullList].fillna(0)
dfTest[numNullList] = dfTest[numNullList].fillna(0)

# fill categorical columns with 'None'
dfTrain[catNullList] = dfTrain[catNullList].fillna("None")
dfVal[catNullList] = dfVal[catNullList].fillna("None")
dfTest[catNullList] = dfTest[catNullList].fillna("None")

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "02_handlemissingvalues")